# NB SAM IT Solutions — Recruitment Data QA

This notebook documents the data-quality and validation analysis carried out
during my Business Analyst internship at NB SAM IT Solutions Pvt. Ltd.

## Data Confidentiality Notice

The following files contain confidential company and candidate information
and **must not be publicly shared or uploaded to GitHub**:

- `Daily Update.xlsx`
- `Daily_Update_Cleaned.xlsx`
- Any other original or intermediate files containing real candidate-level data

The QA analysis uses the company's internal data as a baseline. The original
source files are not included in the public repository.

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the target cleaned dataset
file_path = 'Daily_Update_Cleaned.xlsx'
df = pd.read_excel(file_path, sheet_name='Sheet3')

# Standardize column headers by stripping trailing/leading whitespaces
df.columns = [col.strip() for col in df.columns]

issues = []

# ── 1. Current CTC ≤ Expected CTC ────────────────────────────────────────────
# Ensure we drop any rows where CTC fields are empty before making comparisons
mask = (df['Current CTC'].notna()) & (df['Expected CTC'].notna()) & (df['Current CTC'] > df['Expected CTC'])
flagged = df[mask][['S.No.', 'Name', 'Current CTC', 'Expected CTC']]
if not flagged.empty:
    issues.append(("Current CTC > Expected CTC", flagged))

# ── 2. Relevant Experience ≤ Total Experience ─────────────────────────────────
mask = (df['Relevant Experience (Yrs)'].notna()) & (df['Total Experience (Yrs)'].notna()) & (df['Relevant Experience (Yrs)'] > df['Total Experience (Yrs)'])
flagged = df[mask][['S.No.', 'Name', 'Relevant Experience (Yrs)', 'Total Experience (Yrs)']]
if not flagged.empty:
    issues.append(("Relevant Exp > Total Exp", flagged))

# ── 3. Notice Period not negative ─────────────────────────────────────────────
mask = (df['Notice Period (Days)'].notna()) & (df['Notice Period (Days)'] < 0)
flagged = df[mask][['S.No.', 'Name', 'Notice Period (Days)']]
if not flagged.empty:
    issues.append(("Negative Notice Period", flagged))

# ── 4. Date is valid and parseable ────────────────────────────────────────────
# Use a temporary series to avoid changing underlying DataFrame format prematurely
parsed_date = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
mask = parsed_date.isna() & df['Date'].notna()
flagged = df[mask][['S.No.', 'Name', 'Date']]
if not flagged.empty:
    issues.append(("Invalid or unparseable Date format", flagged))

# ── 5. Client Share Date >= Submission Date (where both exist) ────────────────
parsed_share_date = pd.to_datetime(df['Client Share Date'], dayfirst=True, errors='coerce')
mask = parsed_share_date.notna() & parsed_date.notna() & (parsed_share_date < parsed_date)
flagged = df[mask][['S.No.', 'Name', 'Date', 'Client Share Date']]
if not flagged.empty:
    issues.append(("Client Share Date occurs before Submission Date", flagged))

# ── 6. Allowed values — Profile Status ───────────────────────────────────────
valid_status = ['Shared', 'Internal Reject', 'Rejected by Client (Duplicate)']
# Normalize text comparisons by stripping string values
status_check = df['Profile Status'].astype(str).str.strip()
mask = ~status_check.isin(valid_status) & df['Profile Status'].notna()
flagged = df[mask][['S.No.', 'Name', 'Profile Status']]
if not flagged.empty:
    issues.append(("Invalid Profile Status category detected", flagged))

# ── 7. Allowed values — Client Name ──────────────────────────────────────────
valid_clients = ['Infosys', 'Westernacher', 'Sidra Force', 'N/A (Internal Reject)', 'N/A (Duplicate)']
client_check = df['Client Name'].astype(str).str.strip()
mask = ~client_check.isin(valid_clients) & df['Client Name'].notna()
flagged = df[mask][['S.No.', 'Name', 'Client Name']]
if not flagged.empty:
    issues.append(("Invalid Client Name value detected", flagged))

# ── 8. Allowed values — Referred By ──────────────────────────────────────────
valid_hrs = ['Sharf', 'Masarrat', 'Abhilasha', 'Neha', 'Abhilasha & Masarrat', 'Sharf & Neha']
referred_check = df['Referred By'].astype(str).str.strip()
mask = ~referred_check.isin(valid_hrs) & df['Referred By'].notna() & (referred_check != "")
flagged = df[mask][['S.No.', 'Name', 'Referred By']]
if not flagged.empty:
    issues.append(("Unexpected value in Referred By (Recruiter)", flagged))

# ── 9. Client Database Duplicate is Yes/No only ───────────────────────────────
valid_dup = ['Yes', 'No']
dup_check = df['Client Database Duplicate'].astype(str).str.strip()
mask = ~dup_check.isin(valid_dup) & df['Client Database Duplicate'].notna()
flagged = df[mask][['S.No.', 'Name', 'Client Database Duplicate']]
if not flagged.empty:
    issues.append(("Invalid Duplicate flag (Must be Yes/No)", flagged))

# ── Print report ──────────────────────────────────────────────────────────────
print("=" * 60)
print("DATA VALIDATION QUALITY ASSURANCE REPORT")
print("=" * 60)

if not issues:
    print("\n SUCCESS: All checks passed perfectly, dataset is structurally valid.")
else:
    print(f"\n ATTENTION: Found {len(issues)} structural constraints broken.")
    for title, rows in issues:
        print(f"\n {title} — {len(rows)} row(s) flagged:")
        print("-" * 50)
        print(rows.to_string(index=False))
        print("-" * 50)

print(f"\n{'=' * 60}")
print(f"  Total Validations Monitored : 9")
print(f"  Total Inconsistencies Found : {len(issues)}")
print(f"  Total Records Inspected     : {len(df)}")
print("=" * 60)

DATA VALIDATION QUALITY ASSURANCE REPORT

 ATTENTION: Found 1 structural constraints broken.

 Relevant Exp > Total Exp — 1 row(s) flagged:
--------------------------------------------------
 S.No.         Name  Relevant Experience (Yrs)  Total Experience (Yrs)
    24 Pramod Singh                        7.3                     6.5
--------------------------------------------------

  Total Validations Monitored : 9
  Total Inconsistencies Found : 1
  Total Records Inspected     : 77


In [ ]:
import openpyxl

# 1. Load the spreadsheet workbook
file_path = 'Daily_Update_Cleaned.xlsx'
wb = openpyxl.load_workbook(file_path)
ws = wb['Sheet3']

# 2. Extract column headers to find exact coordinates
headers = [cell.value for cell in ws[1]]
sno_col = headers.index('S.No.') + 1
rel_exp_col = headers.index('Relevant Experience (Yrs)') + 1
tot_exp_col = headers.index('Total Experience (Yrs)') + 1

# 3. Locate the row corresponding to S.No. 24
target_row = None
for row in range(2, ws.max_row + 1):
    if ws.cell(row=row, column=sno_col).value == 24:
        target_row = row
        break

# 4. Perform the surgical swap if the row is found
if target_row:
    # Get current values
    current_relevant = ws.cell(row=target_row, column=rel_exp_col).value
    current_total = ws.cell(row=target_row, column=tot_exp_col).value

    # Swap them
    ws.cell(row=target_row, column=rel_exp_col).value = current_total
    ws.cell(row=target_row, column=tot_exp_col).value = current_relevant

    # Save the file cleanly
    wb.save(file_path)
    wb.close()
    print(f"Success")
else:
    print("Error: S.No. 24 could not be found in the sheet.")

Success


In [ ]:
import os
import openpyxl

old_file = "Daily_Update_Cleaned.xlsx"
new_file = "NB_SAM_Recruitment_Cleaned.xlsx"
old_sheet_name = "Sheet3"
new_sheet_name = "Recruitment_Data"

if os.path.exists(old_file):
    # 1. Open the original workbook
    wb = openpyxl.load_workbook(old_file)

    # 2. Rename the target sheet if it exists
    if old_sheet_name in wb.sheetnames:
        ws = wb[old_sheet_name]
        ws.title = new_sheet_name

        # 3. Save as the brand new file name
        wb.save(new_file)
        wb.close()
        print(f"Dataset saved as: '{new_file}'")
        print(f"Target sheet renamed to: '{new_sheet_name}'")

        # 4. Remove old obsolete file
        os.remove(old_file)
        print(f"Cleaned up old file target: '{old_file}'")
    else:
        wb.close()
        print(f"Error: Sheet '{old_sheet_name}' was not found in the file.")
else:
    print(f"Error: The file '{old_file}' does not exist in your directory.")

Dataset saved as: 'NB_SAM_Recruitment_Cleaned.xlsx'
Target sheet renamed to: 'Recruitment_Data'
Cleaned up old file target: 'Daily_Update_Cleaned.xlsx'


In [ ]:
import pandas as pd

# 1. Load the Excel file from the correct sheet
df = pd.read_excel('NB_SAM_Recruitment_Cleaned.xlsx', sheet_name='Recruitment_Data')

# 2. Safety check: ensure columns are treated as plain text so nothing gets erased or altered
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': '', 'NaN': '', 'NaT': '', 'NAT': '', 'None': ''})

# 3. Export directly to CSV
df.to_csv('NB_SAM_Recruitement_Cleaned.csv', index=False, header=True)
print("Successfully exported to CSV.")

Successfully exported to CSV.


In [ ]:
import pandas as pd
import numpy as np

master = pd.read_csv('/content/NB_SAM_Recruitment_Master.csv')
real  = master.iloc[:77].copy()
synth = master.iloc[77:].copy()

sep = "=" * 65

# ── 1. Recruiter Distribution ─────────────────────────────────────────────
print(sep)
print("  1. RECRUITER DISTRIBUTION")
print(sep)
comp = pd.DataFrame({
    'Real %'     : real['Referred By'].value_counts(normalize=True).mul(100).round(2),
    'Synthetic %': synth['Referred By'].value_counts(normalize=True).mul(100).round(2),
    'Master %'   : master['Referred By'].value_counts(normalize=True).mul(100).round(2),
}).fillna(0)
print(comp)

# ── 2. Skill Distribution ─────────────────────────────────────────────────
print(f"\n{sep}")
print("  2. SKILL DISTRIBUTION (Top 10)")
print(sep)
comp = pd.DataFrame({
    'Real %'     : real['Skill'].value_counts(normalize=True).mul(100).round(2),
    'Synthetic %': synth['Skill'].value_counts(normalize=True).mul(100).round(2),
    'Master %'   : master['Skill'].value_counts(normalize=True).mul(100).round(2),
}).fillna(0).head(10)
print(comp)

# ── 3. Profile Status Distribution ───────────────────────────────────────
print(f"\n{sep}")
print("  3. PROFILE STATUS DISTRIBUTION")
print(sep)
comp = pd.DataFrame({
    'Real %'     : real['Profile Status'].value_counts(normalize=True).mul(100).round(2),
    'Synthetic %': synth['Profile Status'].value_counts(normalize=True).mul(100).round(2),
    'Master %'   : master['Profile Status'].value_counts(normalize=True).mul(100).round(2),
}).fillna(0)
print(comp)

print(synth[synth['Profile Status'] == 'Shared']['Client Name'].value_counts())
print(synth['Client Name'].unique())

# ── 4. Client Distribution (Shared only) ─────────────────────────────────
print(f"\n{sep}")
print("  4. CLIENT DISTRIBUTION (Shared profiles only)")
print(sep)
r = real[real['Profile Status'].str.strip()   == 'Shared']
s = synth[synth['Profile Status'].str.strip() == 'Shared']
m = master[master['Profile Status'].str.strip()== 'Shared']
comp = pd.DataFrame({
    'Real %'     : r['Client Name'].value_counts(normalize=True).mul(100).round(2),
    'Synthetic %': s['Client Name'].value_counts(normalize=True).mul(100).round(2),
    'Master %'   : m['Client Name'].value_counts(normalize=True).mul(100).round(2),
}).fillna(0)
print(comp)

# ── 5. Location Distribution ──────────────────────────────────────────────
print(f"\n{sep}")
print("  5. LOCATION DISTRIBUTION")
print(sep)
comp = pd.DataFrame({
    'Real %'     : real['Location'].value_counts(normalize=True).mul(100).round(2),
    'Synthetic %': synth['Location'].value_counts(normalize=True).mul(100).round(2),
    'Master %'   : master['Location'].value_counts(normalize=True).mul(100).round(2),
}).fillna(0)
print(comp)

# ── 6. Notice Period Distribution ─────────────────────────────────────────
print(f"\n{sep}")
print("  6. NOTICE PERIOD DISTRIBUTION")
print(sep)
comp = pd.DataFrame({
    'Real %'     : real['Notice Period (Days)'].value_counts(normalize=True).mul(100).round(2),
    'Synthetic %': synth['Notice Period (Days)'].value_counts(normalize=True).mul(100).round(2),
    'Master %'   : master['Notice Period (Days)'].value_counts(normalize=True).mul(100).round(2),
}).fillna(0)
print(comp)

# ── 7. Null Rates Per Column ──────────────────────────────────────────────
print(f"\n{sep}")
print("  7. NULL RATES PER COLUMN")
print(sep)
null_cols = [
    'Referred By', 'Location', 'Relevant Experience (Yrs)',
    'Total Experience (Yrs)', 'Notice Period (Days)',
    'Current CTC', 'Expected CTC', 'Phone Number', 'Email',
    'Assigned Recruiter Follow-up'
]
null_comp = pd.DataFrame({
    'Real %'     : real[null_cols].isnull().mean().mul(100).round(2),
    'Synthetic %': synth[null_cols].isnull().mean().mul(100).round(2),
    'Master %'   : master[null_cols].isnull().mean().mul(100).round(2),
})
print(null_comp)

# ── 8. Numeric Stats — Experience & CTC ──────────────────────────────────
print(f"\n{sep}")
print("  8. NUMERIC STATS — EXPERIENCE & CTC")
print(sep)
num_cols = [
    'Relevant Experience (Yrs)', 'Total Experience (Yrs)',
    'Current CTC', 'Expected CTC'
]
for col in num_cols:
    print(f"\n  {col}:")
    stats = pd.DataFrame({
        'Real'     : real[col].describe(),
        'Synthetic': synth[col].describe(),
        'Master'   : master[col].describe(),
    }).round(2)
    print(stats.loc[['mean','50%','min','max','std']])

# ── 9. Summary ────────────────────────────────────────────────────────────
print(f"\n{sep}")
print("  9. ROW COUNT SUMMARY")
print(sep)
print(f"  Real rows       : {len(real)}")
print(f"  Synthetic rows  : {len(synth)}")
print(f"  Master total    : {len(master)}")
print(sep)

  1. RECRUITER DISTRIBUTION
                      Real %  Synthetic %  Master %
Referred By                                        
Masarrat               32.88        29.42     29.75
Sharf                  24.66        28.55     28.18
Abhilasha              24.66        23.77     23.85
Neha                   15.07        15.51     15.47
Abhilasha & Masarrat    1.37         1.45      1.44
Sharf & Neha            1.37         1.30      1.31

  2. SKILL DISTRIBUTION (Top 10)
                                 Real %  Synthetic %  Master %
Skill                                                         
AMAZON REQUIREMENT                 3.90         4.01      4.00
AMAZON REQUIREMENT SOFT DEV ENG    1.30         1.52      1.50
AWS DEVOPS                         3.90         3.87      3.88
COLOCATION INFRASTRUCTURE LEAD     1.30         1.24      1.25
DOT NET DEVELOPER                  6.49         6.64      6.62
FRONT END DEVELOPER                5.19         4.70      4.75
HCM ANALYST OR ADM